# Lab 0b — Python, just enough to start

<a class="btn btn-primary btn-sm" href="https://dhruvbalwada.github.io/intro-climate-modeling-fall2026/labs/lab0b_python_basics.ipynb" download>&#8681; Download this notebook (.ipynb)</a>

*Then upload it to [leap.2i2c.cloud](https://leap.2i2c.cloud/) and open it there.*

This is a quick tour of the four tools we use all semester: **Python**, **numpy** (numbers), **matplotlib** (plots), and a first look at **pandas / xarray** (labeled data). Run every cell, change things, break things.

If you already know all this — skim it, then help a neighbor.

## 0. Notebooks

A notebook is a stack of **cells**. A cell is either text (like this) or code (like the next one). `Shift+Enter` runs the current cell and moves on. Cells share memory — a variable made in one cell exists in the next.

In [ ]:
print("hello, climate")
2 + 2   # the last line's value is displayed automatically

## 1. Variables, and doing arithmetic with them

In [ ]:
S0 = 1361.0        # solar constant, W/m2   (a float)
albedo = 0.3       # fraction reflected     (a float)
planet = "Earth"   # a string

absorbed = S0 * (1 - albedo) / 4
print(planet, "absorbs", absorbed, "W/m2")
print(f"{planet} absorbs {absorbed:.1f} W/m2")   # f-string: nicer formatting

**Your turn:** change the albedo to 0.75 (roughly Venus) and re-run. Does absorbed energy go up or down?

## 2. Functions

A function packages a calculation so you can reuse it.

In [ ]:
def absorbed_solar(S0, albedo):
    """Solar energy absorbed per square metre of planet surface [W/m2]."""
    return S0 * (1 - albedo) / 4

print(absorbed_solar(1361, 0.30))   # Earth
print(absorbed_solar(2601, 0.75))   # Venus

## 3. Loops

For repeating something — including *stepping a model forward in time*, which is what we do in Lab 0d.

In [ ]:
for year in [2020, 2021, 2022]:
    print(year, "→", year - 1850, "years since the pre-industrial baseline")

# a list, built up inside a loop
squares = []
for i in range(5):        # range(5) gives 0,1,2,3,4
    squares.append(i**2)
print(squares)

## 4. numpy — arrays of numbers

Lists are fine, but for data we use **numpy arrays**: math applies to the whole array at once (no loop needed), and they're fast.

In [ ]:
import numpy as np

temps = np.array([12.3, 15.1, 9.8, 21.0, 18.4])   # °C

print("array:      ", temps)
print("in Kelvin:  ", temps + 273.15)     # applies to every element
print("mean:       ", temps.mean())
print("std dev:    ", temps.std())
print("max:        ", temps.max())
print("first two:  ", temps[:2])          # indexing starts at 0
print("above 15:   ", temps[temps > 15])  # selecting by condition

In [ ]:
# arrays you generate rather than type out
days = np.arange(0, 365)              # 0,1,2,...,364
x = np.linspace(0, 2*np.pi, 5)        # 5 evenly spaced values from 0 to 2pi

print(days[:5], "...", days[-3:])
print(np.round(np.sin(x), 3))         # math functions work on whole arrays

## 5. matplotlib — making a figure

Every plot you make this semester should have **axis labels with units**. Really.

In [ ]:
import matplotlib.pyplot as plt

day = np.arange(365)
T = 12.5 + 11.0 * np.cos(2 * np.pi * (day - 201) / 365)   # a fake seasonal cycle

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(day, T, label="model")
ax.set_xlabel("day of year")
ax.set_ylabel("temperature [°C]")
ax.set_title("a made-up seasonal cycle")
ax.legend();

In [ ]:
# two other plot types you'll need constantly
noise = np.random.default_rng(0).normal(0, 4, size=365)   # fake "weather"

fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].plot(day, T + noise, lw=0.8)        # line
ax[0].set_xlabel("day"); ax[0].set_ylabel("°C"); ax[0].set_title("with weather")
ax[1].hist(T + noise, bins=30)            # histogram — a distribution!
ax[1].set_xlabel("°C"); ax[1].set_title("its distribution")
plt.tight_layout()

**Your turn:** make the noise bigger (change `4` to `10`) and re-run. What happens to the histogram — and what would that mean for a place's climate?

## 6. pandas — tables with labels

Real data comes with labels (dates, station names). `pandas` handles tables; we use it to read CSV files, including straight from a URL.

In [ ]:
import pandas as pd

url = "https://dhruvbalwada.github.io/intro-climate-modeling-fall2026/labs/data/centralpark_obs_era5_1995-2014.csv"
df = pd.read_csv(url, index_col=0, parse_dates=True)

print(df.shape, "rows × columns")
df.head()

In [ ]:
print(df["t_obs"].mean())          # one column, its mean
df["t_obs"]["2010"].plot(figsize=(8, 2.5), lw=0.8)   # slice by date, and plot
plt.ylabel("°C");

## 7. xarray — labeled data with *dimensions*

Climate model output isn't a table: it's temperature on (time × latitude × longitude), often many gigabytes. `xarray` is pandas for that world — you index by **name**, not by number. We'll use it properly later in the course; here's the flavor.

In [ ]:
import xarray as xr

# a small fake "model output": temperature over 12 months and 5 latitudes
lat = np.array([-60, -30, 0, 30, 60])
month = np.arange(1, 13)
data = 30 * np.cos(np.deg2rad(lat))[None, :] + 5 * np.sin(2*np.pi*(month[:, None]-1)/12)

da = xr.DataArray(data, dims=["month", "lat"],
                  coords={"month": month, "lat": lat},
                  name="temperature", attrs={"units": "degC"})
da

In [ ]:
print("equator, July:", float(da.sel(lat=0, month=7)), "°C")   # select BY NAME
print("annual mean at 60N:", float(da.sel(lat=60).mean("month")), "°C")

da.plot(figsize=(6, 3));   # xarray knows the labels, so it labels the plot for you

## That's enough to start

You now have everything needed for the rest of today: arrays, a plot, a table read from a URL, and a loop.

**Two habits worth keeping all semester:**

1. Label your axes, with units.
2. When something breaks, read the *last line* of the error first — it usually says exactly what's wrong. AI assistants are also good at explaining errors; use them, and make sure you understand the fix.

**Want to go deeper?** A fuller treatment of Python for earth and environmental data — numpy, pandas, xarray, plotting, working with real datasets — lives in the companion course site: <https://earth-ds-ml.github.io/summer_2026/intro.html>